In [1]:
import os
os.chdir('../../../..')

In [2]:
import polars as pl

from src.helper_functions import create_chemiscope_viewer, average_numeric_by_cluster
from src.datasets import QM9Dataset
from src.outliers.outlier_detection import hdbscan_outliers, knn_outliers, lof_outliers
from src.outliers.eval_outlier_results import plot_score_distributions, evaluate_outlier_methods
from src.non_euclidean import Riemann, Grassmann, Wasserstein

INFO: Enabling RDKit 2025.09.4 jupyter extensions


In [3]:
def keep_outliers(outliers_df, n):
    # determines how many too keep of each
    return outliers_df.group_by("outlier_category", maintain_order=True).head(n)
    
outliers = pl.read_parquet('data/QM9/outliers/synthetic_outliers.parquet')
outliers = keep_outliers(outliers, 10)

In [4]:
descriptor = "soap"

qm9 = QM9Dataset(limit=1000, 
                 sampling_strategy="stratified",
                 stratify_by=["num_atoms", "gap"], 
                 descriptors=["soap"],
                 injected_molecules = outliers,
                 )
df = qm9.load()
soap_matrix = qm9.get_descriptor_matrices("soap")

2026-04-26 13:22:52.468 | INFO     | src.datasets:load:795 - Loading cached full QM9 dataset from: data/QM9/dataset_cleaned.parquet
2026-04-26 13:22:52.642 | INFO     | src.datasets:_sample_qm9_df:995 - QM9 sampling complete: strategy=stratified, requested_limit=1100, returned_rows=1100.
2026-04-26 13:22:56.015 | INFO     | src.datasets:_add_requested_descriptors:187 - Applying requested QM9 descriptors to sampled dataframe (rows=1140).
2026-04-26 13:22:56.078 | INFO     | src.features:compute_soap:193 - Computing SOAP (rcut=6.0, nmax=8, lmax=6)...
2026-04-26 13:23:04.152 | SUCCESS  | src.datasets:add_soap:1147 - Added SOAP embeddings.
2026-04-26 13:23:04.153 | INFO     | src.datasets:_add_requested_descriptors:212 - Added descriptor column(s): ['soap_embedding']
2026-04-26 13:23:04.153 | INFO     | src.datasets:inject_outliers:783 - Injected custom outliers into QM9 dataframe: requested=40, injected=40, total_rows=1140.
2026-04-26 13:23:04.153 | INFO     | src.datasets:_add_requested_

In [5]:
riemann = Riemann()
dist_matrix = riemann.distance_matrix(feature_matrices=soap_matrix, metric='affine-invariant', n_pca=30)

2026-04-26 13:23:19.815 | INFO     | src.non_euclidean:distance_matrix:648 - Computing Riemann distance matrix | Features: invariant | Metric: affine-invariant
2026-04-26 13:23:19.816 | INFO     | src.non_euclidean:_get_spd_matrices:600 - Applying PCA to reduce feature dimension to 30...
Computing Distances: 100%|██████████| 1040/1040 [00:21<00:00, 49.42it/s]


In [6]:
grassmann = Grassmann()
dist_matrix_grass = grassmann.distance_matrix(precomputed_feature_matrices=soap_matrix)

2026-04-26 13:23:43.346 | INFO     | src.non_euclidean:distance_matrix:557 - Computing Grassmann distance matrix for 1040 items (k=3, method='svd', features='invariant', normalized=True).
Grassmann distances: 100%|██████████| 1040/1040 [03:32<00:00,  4.91pair/s]


In [7]:
wasserstein = Wasserstein()
dist_matrix_wasserstein = wasserstein.distance_matrix(feature_matrices=soap_matrix)

2026-04-26 13:27:15.383 | INFO     | src.non_euclidean:distance_matrix:309 - Computing Wasserstein distance matrix | Features: invariant
Wasserstein (invariant): 100%|██████████| 540280/540280 [02:11<00:00, 4120.50pair/s]


In [9]:
df_riemann = df.clone()
df_riemann = hdbscan_outliers(df, dist_matrix)
df_riemann = knn_outliers(df_riemann, dist_matrix)
df_riemann = lof_outliers(df_riemann, dist_matrix)

Starting HDBSCAN Auto-Tuning over 122 combinations for N=1040...


🔍 Evaluating HDBSCAN: 100%|██████████| 122/122 [00:02<00:00, 41.20cfg/s, mcs=104, ms=104, clusters=0]



--- Auto-Tuning Results ---
Selected params: min_cluster_size=3, min_samples=2
Metrics: clusters=2, noise=0.0433, persistence=0.020
HDBSCAN — 3 distinct labels: -1: 45, 0: 992, 1: 3
k-NN — 2 distinct: -1: 104, 1: 936
LOF — 1 distinct: 1: 1040


In [10]:
results = evaluate_outlier_methods(df_riemann, ["hdbscan", "knn", "lof"])
results

Method,Global_Recall,False_Positive_Rate,Flagged_QM9_Count,Total_Flagged,Total_Missed,ROC_AUC,Recall: element_outliers,Recall: extreme_outliers,Recall: size_outliers,Recall: topology_outliers
str,f64,f64,i64,i64,i64,f64,f64,f64,f64,f64
"""HDBSCAN""",0.0,0.045,45,0,40,0.266,0.0,0.0,0.0,0.0
"""KNN""",0.0,0.104,104,0,40,0.361,0.0,0.0,0.0,0.0
"""LOF""",0.0,0.0,0,0,40,0.279,0.0,0.0,0.0,0.0


In [12]:
df_wasserstein = df.clone()
df_wasserstein = hdbscan_outliers(df_wasserstein, dist_matrix_wasserstein)
df_wasserstein = knn_outliers(df_wasserstein, dist_matrix_wasserstein)
df_wasserstein = lof_outliers(df_wasserstein, dist_matrix_wasserstein)

Starting HDBSCAN Auto-Tuning over 122 combinations for N=1040...


🔍 Evaluating HDBSCAN: 100%|██████████| 122/122 [00:03<00:00, 39.54cfg/s, mcs=104, ms=104, clusters=0]



--- Auto-Tuning Results ---
Selected params: min_cluster_size=10, min_samples=1
Metrics: clusters=2, noise=0.1817, persistence=0.037
HDBSCAN — 3 distinct labels: -1: 189, 0: 841, 1: 10
k-NN — 2 distinct: -1: 104, 1: 936
LOF — 2 distinct: -1: 21, 1: 1019


In [13]:
results = evaluate_outlier_methods(df_wasserstein, ["hdbscan", "knn", "lof"])
results

Method,Global_Recall,False_Positive_Rate,Flagged_QM9_Count,Total_Flagged,Total_Missed,ROC_AUC,Recall: extreme_outliers,Recall: size_outliers,Recall: element_outliers,Recall: topology_outliers
str,f64,f64,i64,i64,i64,f64,f64,f64,f64,f64
"""HDBSCAN""",0.7,0.161,161,28,12,0.847,0.8,0.8,0.7,0.5
"""KNN""",0.775,0.073,73,31,9,0.927,0.9,0.7,0.8,0.7
"""LOF""",0.475,0.002,2,19,21,0.977,0.8,0.2,0.5,0.4


In [14]:
df_grassmann = df.clone()
df_grassmann = hdbscan_outliers(df_grassmann, dist_matrix_grass)
df_grassmann = knn_outliers(df_grassmann, dist_matrix_grass)
df_grassmann = lof_outliers(df_grassmann, dist_matrix_grass)

Starting HDBSCAN Auto-Tuning over 122 combinations for N=1040...


🔍 Evaluating HDBSCAN: 100%|██████████| 122/122 [00:03<00:00, 38.78cfg/s, mcs=104, ms=104, clusters=0]


--- Auto-Tuning Results ---
Selected params: min_cluster_size=3, min_samples=4
Metrics: clusters=2, noise=0.1519, persistence=0.076
HDBSCAN — 3 distinct labels: -1: 158, 0: 879, 1: 3
k-NN — 2 distinct: -1: 104, 1: 936
LOF — 2 distinct: -1: 30, 1: 1010


In [15]:
results = evaluate_outlier_methods(df_grassmann, ["hdbscan", "knn", "lof"])
results

Method,Global_Recall,False_Positive_Rate,Flagged_QM9_Count,Total_Flagged,Total_Missed,ROC_AUC,Recall: element_outliers,Recall: extreme_outliers,Recall: size_outliers,Recall: topology_outliers
str,f64,f64,i64,i64,i64,f64,f64,f64,f64,f64
"""HDBSCAN""",0.9,0.122,122,36,4,0.94,1.0,0.9,0.7,1.0
"""KNN""",0.85,0.07,70,34,6,0.936,1.0,0.8,0.6,1.0
"""LOF""",0.6,0.006,6,24,16,0.961,0.9,0.5,0.1,0.9


In [ ]:
def append_outlier_consensus(df: pl.DataFrame) -> pl.DataFrame:
    """Calculates outlier consensus and creates categorical strings for Chemiscope visualization."""
    models = ["hdbscan", "knn", "lof"]
    cols_to_add = []
    vote_cols = []
    
    for model in models:
        col_name = f"{model}_label"
        if col_name in df.columns:
            # 1. Create binary flag for voting (1 = outlier, 0 = inlier)
            vote_col = f"is_{model}_outlier"
            cols_to_add.append((pl.col(col_name) == -1).cast(pl.Int8).alias(vote_col))
            vote_cols.append(pl.col(vote_col))
            
            # 2. Create readable string category for individual viewing
            cols_to_add.append(
                pl.when(pl.col(col_name) == -1).then(pl.lit("Outlier"))
                .otherwise(pl.lit("Inlier"))
                .alias(f"{model}_category")
            )
            
    df_out = df.with_columns(cols_to_add)
    
    # 3. Calculate total votes and map to color categories
    if vote_cols:
        df_out = df_out.with_columns(pl.sum_horizontal(vote_cols).alias("outlier_votes"))
        
        df_out = df_out.with_columns(
            pl.when(pl.col("outlier_votes") == 3).then(pl.lit("3: Consensus Outlier"))
            .when(pl.col("outlier_votes") == 2).then(pl.lit("2: Strong Outlier"))
            .when(pl.col("outlier_votes") == 1).then(pl.lit("1: Weak Outlier"))
            .otherwise(pl.lit("0: Inlier"))
            .alias("Consensus_Category")
        )
        
    return df_out

In [ ]:
df_for_viewer = append_outlier_consensus(df)
create_chemiscope_viewer(df_for_viewer, dist_matrix, df_for_viewer['is_injected'].to_numpy(), 'ISOMAP')

Running ISOMAP dimensionality reduction...
Converting structures/molecules to ASE Atoms for Chemiscope...
Assembling properties for Chemiscope...
Generating Chemiscope widget...
Saved Chemiscope input to: qm9_ISOMAP_clustering.json
If the viewer does not open automatically, run `chemiscope show qm9_ISOMAP_clustering.json`.


<ChemiscopeWidget(meta={'name': 'QM9 - ISOMAP Clustering'}, settings={'map': {'x': {'property': 'ISOMAP_1'}, '…